In [ ]:
from collections import OrderedDict
from datetime import date, datetime, time, timedelta
import html
import math

import ipywidgets as widgets
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, clear_output, display as ipy_display
from ipywidgets import SelectionRangeSlider
from matplotlib.lines import Line2D
from matplotlib.transforms import blended_transform_factory
from pyspark.sql import functions as F
from pyspark.sql.window import Window

ASSET_TABLE = "ehm_fleetstore_prd1eun82719_internal.assetmanagement.aircraftengine"
MASTER_TABLE = "ehm_fleetstore_prd1eun82719_internal.`g700-pearl700`.`emucontinuousmaster-aircraftengine-cda`"
SCAN_TABLE = "ehm_fleetstore_prd1eun82719_internal.`g700-pearl700`.`emucontinuousscan-cda`"
ENGINE_TYPE_CODE = "Pearl 700"
CHANNELS = ("AC", "BC")
PARAMETER_PREFIX = "Parameter:EVHMU_EEC_"

for legacy_widget_name in (
    "AircraftID",
    "EngineSerialNumber",
    "Flight_Date",
    "num_years",
    "FlightStartDateTime",
):
    try:
        dbutils.widgets.remove(legacy_widget_name)
    except Exception:
        pass

asset_df = (
    spark.table(ASSET_TABLE)
    .where(F.col("EngineTypeCode") == ENGINE_TYPE_CODE)
    .select(
        F.col("LatestAircraftLatestIdentifier").cast("string").alias("AircraftIdentifier"),
        F.col("LatestAircraftId").alias("AircraftId"),
        F.col("EngineSerialNumber").cast("string").alias("EngineSerialNumber"),
        F.col("EngineId").alias("EngineId"),
        F.col("LatestOperatorName").alias("OperatorName"),
    )
    .where(
        F.col("AircraftIdentifier").isNotNull()
        & F.col("AircraftId").isNotNull()
        & F.col("EngineSerialNumber").isNotNull()
    )
    .dropDuplicates()
)

asset_rows = asset_df.collect()
aircraft_options = sorted({row.AircraftIdentifier for row in asset_rows})
if not aircraft_options:
    raise RuntimeError(f"No aircraft were found for engine type {ENGINE_TYPE_CODE}.")

asset_aircraft_ids_by_identifier = {}
asset_esns_by_aircraft = {}
for row in asset_rows:
    asset_aircraft_ids_by_identifier.setdefault(row.AircraftIdentifier, set()).add(row.AircraftId)
    asset_esns_by_aircraft.setdefault(row.AircraftIdentifier, set()).add(str(row.EngineSerialNumber))

print(f"Loaded {len(aircraft_options)} aircraft with {ENGINE_TYPE_CODE} engines.")

In [ ]:
def _calendar_bounds(start_date, end_date):
    start_boundary = datetime.combine(start_date, time.min)
    end_boundary = datetime.combine(end_date + timedelta(days=1), time.min)
    return start_boundary, end_boundary


def _month_partitions_between_dates(start_date, end_date):
    current = date(start_date.year, start_date.month, 1)
    final = date(end_date.year, end_date.month, 1)
    partitions = []
    while current <= final:
        partitions.append((current.year, current.month))
        current = date(current.year + 1, 1, 1) if current.month == 12 else date(current.year, current.month + 1, 1)
    return partitions


def _position_sort_value(position):
    if position is None:
        return (2, "")
    try:
        return (0, float(position))
    except Exception:
        return (1, str(position))


def _ordered_esns(engine_positions):
    return sorted(
        engine_positions,
        key=lambda esn: (_position_sort_value(engine_positions.get(esn)), str(esn)),
    )


def _resolve_engine_windows(aircraft_identifier, start_boundary, end_boundary):
    master_df = (
        spark.table(MASTER_TABLE)
        .where(
            (F.col("AircraftIdentifier").cast("string") == str(aircraft_identifier))
            & F.col("StartDatetime").isNotNull()
            & F.col("EndDatetime").isNotNull()
            & (F.col("StartDatetime") < F.lit(end_boundary).cast("timestamp"))
            & (F.col("EndDatetime") >= F.lit(start_boundary).cast("timestamp"))
            & F.col("EngineSerialNumber").isNotNull()
            & F.col("AircraftId").isNotNull()
        )
    )
    position_column = (
        F.col("EnginePosition").cast("string")
        if "EnginePosition" in master_df.columns
        else F.lit(None).cast("string")
    )
    windows_df = (
        master_df.select(
            F.col("AircraftId").alias("AircraftId"),
            F.col("EngineSerialNumber").cast("string").alias("EngineSerialNumber"),
            position_column.alias("EnginePosition"),
            F.col("StartDatetime").alias("StartDatetime"),
            F.col("EndDatetime").alias("EndDatetime"),
        )
        .groupBy("AircraftId", "EngineSerialNumber", "StartDatetime")
        .agg(
            F.max("EndDatetime").alias("EndDatetime"),
            F.first("EnginePosition", ignorenulls=True).alias("EnginePosition"),
        )
        .where(F.col("EndDatetime") >= F.col("StartDatetime"))
    )
    descriptor_rows = windows_df.select("EngineSerialNumber", "EnginePosition").distinct().collect()
    position_candidates = {}
    for row in descriptor_rows:
        esn = str(row.EngineSerialNumber)
        position_candidates.setdefault(esn, []).append(row.EnginePosition)
    engine_positions = {
        esn: sorted(positions, key=_position_sort_value)[0]
        for esn, positions in position_candidates.items()
    }
    esns = _ordered_esns(engine_positions)
    aircraft_ids = [row.AircraftId for row in windows_df.select("AircraftId").distinct().collect()]
    return master_df, windows_df, esns, engine_positions, aircraft_ids


def _windowed_scan(
    windows_df,
    aircraft_ids,
    esns,
    requested_columns=None,
    strict_start=None,
    strict_end=None,
):
    scan_source = spark.table(SCAN_TABLE)
    scan_columns = scan_source.columns
    required_keys = ["Timestamp", "AircraftId", "AssetIdentifier"]
    missing_keys = [column for column in required_keys if column not in scan_columns]
    if missing_keys:
        raise ValueError(f"The scan table is missing required columns: {missing_keys}")

    if requested_columns is None:
        selected_scan_columns = list(scan_columns)
    else:
        selected_scan_columns = list(
            dict.fromkeys(required_keys + [column for column in requested_columns if column in scan_columns])
        )

    if strict_start is not None and strict_end is not None:
        partition_start = strict_start
        partition_end = strict_end - timedelta(microseconds=1)
    else:
        bounds = windows_df.agg(
            F.min("StartDatetime").alias("RangeStart"),
            F.max("EndDatetime").alias("RangeEnd"),
        ).first()
        partition_start = bounds.RangeStart
        partition_end = bounds.RangeEnd

    if partition_start is None or partition_end is None or not esns or not aircraft_ids:
        empty_columns = list(dict.fromkeys(selected_scan_columns + ["StartDatetime", "EnginePosition"]))
        expressions = []
        for column in empty_columns:
            if column in scan_columns:
                expressions.append(F.col(f"`{column}`"))
            elif column == "StartDatetime":
                expressions.append(F.lit(None).cast("timestamp").alias(column))
            else:
                expressions.append(F.lit(None).cast("string").alias(column))
        return scan_source.limit(0).select(*expressions), scan_columns

    partition_filter = F.lit(False)
    for partition_year, partition_month in _month_partitions_between_dates(
        partition_start.date(), partition_end.date()
    ):
        partition_filter = partition_filter | (
            (F.col("year") == partition_year) & (F.col("Month") == partition_month)
        )

    scan_filtered = scan_source.where(
        F.col("AircraftId").isin(aircraft_ids)
        & F.col("AssetIdentifier").cast("string").isin(esns)
        & partition_filter
    )
    if strict_start is not None and strict_end is not None:
        scan_filtered = scan_filtered.where(
            (F.col("Timestamp") >= F.lit(strict_start).cast("timestamp"))
            & (F.col("Timestamp") < F.lit(strict_end).cast("timestamp"))
        )

    scan_alias = (
        scan_filtered.select(*[F.col(f"`{column}`") for column in selected_scan_columns])
        .withColumn("_ScanRowId", F.monotonically_increasing_id())
        .alias("scan")
    )
    window_alias = F.broadcast(
        windows_df.select(
            F.col("AircraftId").alias("_WindowAircraftId"),
            F.col("EngineSerialNumber").cast("string").alias("_WindowESN"),
            F.col("EnginePosition").cast("string").alias("_WindowEnginePosition"),
            F.col("StartDatetime").alias("_WindowStartDatetime"),
            F.col("EndDatetime").alias("_WindowEndDatetime"),
        )
    ).alias("window")
    joined = scan_alias.join(
        window_alias,
        (scan_alias["AircraftId"] == window_alias["_WindowAircraftId"])
        & (scan_alias["AssetIdentifier"].cast("string") == window_alias["_WindowESN"])
        & scan_alias["Timestamp"].between(
            window_alias["_WindowStartDatetime"], window_alias["_WindowEndDatetime"]
        ),
        "inner",
    )
    assignment_window = Window.partitionBy(scan_alias["_ScanRowId"]).orderBy(
        window_alias["_WindowStartDatetime"].desc(),
        window_alias["_WindowEndDatetime"].asc(),
    )
    assigned = joined.withColumn("_WindowRank", F.row_number().over(assignment_window)).where(
        F.col("_WindowRank") == 1
    )

    output_expressions = []
    start_inserted = False
    position_inserted = False
    for column in selected_scan_columns:
        if column == "StartDatetime":
            output_expressions.append(window_alias["_WindowStartDatetime"].alias("StartDatetime"))
            start_inserted = True
        elif column == "EnginePosition":
            output_expressions.append(window_alias["_WindowEnginePosition"].alias("EnginePosition"))
            position_inserted = True
        else:
            output_expressions.append(scan_alias[column])
    if not start_inserted:
        output_expressions.append(window_alias["_WindowStartDatetime"].alias("StartDatetime"))
    if not position_inserted:
        output_expressions.append(window_alias["_WindowEnginePosition"].alias("EnginePosition"))
    return assigned.select(*output_expressions), scan_columns


def _latest_date_for_aircraft(aircraft_identifier):
    row = (
        spark.table(MASTER_TABLE)
        .where(F.col("AircraftIdentifier").cast("string") == str(aircraft_identifier))
        .agg(F.max("CalendarId").alias("LatestCalendarId"))
        .first()
    )
    latest_calendar_id = row.LatestCalendarId if row else None
    return (
        datetime.strptime(str(int(latest_calendar_id)), "%Y%m%d").date()
        if latest_calendar_id is not None
        else date.today()
    )


existing_aircraft_widget = globals().get("aircraft_widget")
existing_start_widget = globals().get("start_date_widget")
existing_end_widget = globals().get("end_date_widget")
previous_aircraft = getattr(existing_aircraft_widget, "value", None)
default_aircraft = previous_aircraft if previous_aircraft in aircraft_options else aircraft_options[0]
previous_start_date = getattr(existing_start_widget, "value", None)
previous_end_date = getattr(existing_end_widget, "value", None)
if previous_start_date is None or previous_end_date is None:
    previous_end_date = _latest_date_for_aircraft(default_aircraft)
    previous_start_date = previous_end_date - timedelta(days=6)

aircraft_widget = widgets.Dropdown(
    options=aircraft_options,
    value=default_aircraft,
    description="Aircraft:",
    layout=widgets.Layout(width="420px"),
)
start_date_widget = widgets.DatePicker(
    value=previous_start_date,
    description="Start date:",
    layout=widgets.Layout(width="300px"),
)
end_date_widget = widgets.DatePicker(
    value=previous_end_date,
    description="End date:",
    layout=widgets.Layout(width="300px"),
)
load_flights_btn = widgets.Button(
    description="Load selected flights",
    button_style="primary",
    layout=widgets.Layout(width="220px"),
)
selection_output = widgets.Output()


def _load_selected_flights(_, raise_errors=False):
    load_flights_btn.disabled = True
    try:
        with selection_output:
            clear_output(wait=True)
            chosen_aircraft = aircraft_widget.value
            chosen_start_date = start_date_widget.value
            chosen_end_date = end_date_widget.value
            if chosen_start_date is None or chosen_end_date is None:
                raise ValueError("Both start and end dates are required.")
            if chosen_start_date > chosen_end_date:
                raise ValueError("Start date must not be later than end date.")

            start_boundary, end_boundary = _calendar_bounds(chosen_start_date, chosen_end_date)
            master_df, windows_df, esns, engine_positions, aircraft_ids = _resolve_engine_windows(
                chosen_aircraft, start_boundary, end_boundary
            )
            flight_rows = windows_df.orderBy("StartDatetime", "EngineSerialNumber").collect()
            if flight_rows:
                chosen_cda, scan_columns = _windowed_scan(
                    windows_df,
                    aircraft_ids,
                    esns,
                )
                selected_start = min(row.StartDatetime for row in flight_rows)
                selected_end = max(row.EndDatetime for row in flight_rows)
            else:
                scan_source = spark.table(SCAN_TABLE)
                chosen_cda = scan_source.limit(0)
                scan_columns = scan_source.columns
                selected_start = None
                selected_end = None

            globals().update(
                {
                    "selected_acid": chosen_aircraft,
                    "selected_esns": esns,
                    "engine_positions": engine_positions,
                    "internal_acids": aircraft_ids,
                    "window_start_date": chosen_start_date,
                    "window_end_date": chosen_end_date,
                    "cda_master_dt": master_df,
                    "flight_windows_df": windows_df,
                    "flight_rows": flight_rows,
                    "flight_available": bool(flight_rows),
                    "selected_start_dt": selected_start,
                    "selected_end_dt": selected_end,
                    "start_dt": selected_start,
                    "end_dt": selected_end,
                    "scan_table": spark.table(SCAN_TABLE),
                    "cda_cols": chosen_cda.columns,
                    "cda": chosen_cda,
                    "selection_revision": globals().get("selection_revision", 0) + 1,
                }
            )

            print(f"Aircraft: {chosen_aircraft}")
            print(f"Calendar range: {chosen_start_date} through {chosen_end_date}")
            print(f"ESNs found in overlapping master records: {', '.join(esns) if esns else 'none'}")
            print(f"Engine-flight windows loaded: {len(flight_rows)}")
            if not flight_rows:
                print("WARNING: No overlapping flight windows were found.")
            missing_from_current_assets = sorted(set(esns) - asset_esns_by_aircraft.get(chosen_aircraft, set()))
            if missing_from_current_assets:
                print(
                    "NOTICE: Historical ESNs not present in the latest asset mapping: "
                    + ", ".join(missing_from_current_assets)
                )
    except Exception as exc:
        with selection_output:
            print(f"Unable to load the selected flights: {exc}")
        if raise_errors:
            raise
    finally:
        load_flights_btn.disabled = False


load_flights_btn.on_click(_load_selected_flights)
selection_controls = widgets.VBox(
    [
        aircraft_widget,
        widgets.HBox([start_date_widget, end_date_widget, load_flights_btn]),
        selection_output,
    ]
)
ipy_display(selection_controls)
_load_selected_flights(None, raise_errors=True)

In [ ]:
THRUST_REVERSER_SUFFIXES = [
    "IAcThrustReverser_tcmIntlV_data",
    "IAcThrustReverser_trLeftDoorSwLocked_LOWER__data",
    "IAcThrustReverser_trLeftDoorSwLocked_UPPER__data",
    "IAcThrustReverser_trLwrDoorLeftSwLocked_data",
    "IAcThrustReverser_trLwrDoorRightSwLocked_data",
    "IAcThrustReverser_trLwrDoorRightSwLocked_flt",
    "IAcThrustReverser_trRightDoorSwLocked_LOWER__data",
    "IAcThrustReverser_trRightDoorSwLocked_UPPER__data",
    "IAcThrustReverser_trUprDoorLeftSwLocked_data",
    "IAcThrustReverser_trUprDoorLeftSwLocked_flt",
    "IAcThrustReverser_trUprDoorRightSwLocked_data",
    "IAcThrustReverser_trUprDoorRightSwLocked_flt",
    "IOSThrustReverser_mainTestEnabled_data",
    "IOSThrustReverser_trDoorPosVa_data",
    "IOSThrustReverser_trDoorPosVb_data",
    "IOSThrustReverser_trDoorPosVex_data",
    "IOSThrustReverser_trDoorPosVex_extflt",
    "IOSThrustReverser_trDoorPosVex_intflt",
    "IOSThrustReverser_trLoSwRaw_data",
    "IOSThrustReverser_trUpSwRaw_data",
    "IOtherProcThrustReverser_trLoLVTSelected_data",
    "IOtherProcThrustReverser_trUpLVTSelected_data",
    "IThrustReverser_loOwnTRLVTRaw_data",
    "IThrustReverser_loOwnTRLVTRngFlt_data",
    "IThrustReverser_trAnomaly_data",
    "IThrustReverser_trArmed_data",
    "IThrustReverser_trDeploySelected_data",
    "IThrustReverser_trDoorDeplStatus_data",
    "IThrustReverser_trDoorPosLost_data",
    "IThrustReverser_trDoorStowStatus_data",
    "IThrustReverser_trInTransit_data",
    "IThrustReverser_trInadvDeploy_data",
    "IThrustReverser_trIsDeployed_data",
    "IThrustReverser_trIsUnlocked_data",
    "IThrustReverser_trJam_data",
    "IThrustReverser_trLVTPosVCtrl_data",
    "IThrustReverser_trLegalDeployCmd_data",
    "IThrustReverser_trLessDeplPos_data",
    "IThrustReverser_trLoLVTSelected_data",
    "IThrustReverser_trLoLVTSelected_flt",
    "IThrustReverser_trMTESTimeExpired_data",
    "IThrustReverser_trMTESV_data",
    "IThrustReverser_trMoreDeplPos_data",
    "IThrustReverser_trUnavailable_data",
    "IThrustReverser_trUpLVTSelected_data",
    "IThrustReverser_trUpLVTSelected_flt",
    "IThrustReverser_trVPrSwVFlt_data",
    "IThrustReverser_trVPrSwV_data",
    "IThrustReverser_upOwnTRLVTRaw_data",
    "IThrustReverser_upOwnTRLVTRngFlt_data",
]

THRUST_REVERSER_COLUMNS = [
    f"{PARAMETER_PREFIX}{channel}_{suffix}"
    for channel in CHANNELS
    for suffix in THRUST_REVERSER_SUFFIXES
]
CONTEXT_COLUMNS = [
    f"{PARAMETER_PREFIX}AC_IHPShaft_nhV_data",
    f"{PARAMETER_PREFIX}BC_IHPShaft_nhV_data",
    f"{PARAMETER_PREFIX}AC_IAircraftState_altitudeC_data",
    f"{PARAMETER_PREFIX}BC_IAircraftState_altitudeC_data",
]
BASE_IDENTIFIER_COLUMNS = [
    "Timestamp",
    "StartDatetime",
    "AircraftId",
    "ParentAssetIdentifier",
    "AssetIdentifier",
    "EnginePosition",
]

scan_schema = set(cda_cols)
phase_candidates = [column for column in cda_cols if "flightphase" in column.lower()]
FLIGHT_PHASE_COLUMNS = {}
for channel in CHANNELS:
    channel_prefix = f"{PARAMETER_PREFIX}{channel}_".lower()
    channel_matches = sorted(
        column for column in phase_candidates if column.lower().startswith(channel_prefix)
    )
    if len(channel_matches) == 1:
        FLIGHT_PHASE_COLUMNS[channel] = channel_matches[0]
    elif len(channel_matches) > 1:
        print(f"WARNING: Multiple {channel} FlightPhase columns were found and omitted: {channel_matches}")

IDENTIFIER_COLUMNS = [
    column
    for column in BASE_IDENTIFIER_COLUMNS + list(FLIGHT_PHASE_COLUMNS.values())
    if column in scan_schema
]
missing_core_identifiers = [
    column for column in ("Timestamp", "StartDatetime", "AssetIdentifier") if column not in scan_schema
]
if missing_core_identifiers:
    raise ValueError(f"The selected scan data is missing required columns: {missing_core_identifiers}")

requested_parameter_columns = THRUST_REVERSER_COLUMNS + CONTEXT_COLUMNS
unavailable_parameter_columns = [
    column for column in requested_parameter_columns if column not in scan_schema
]
PARAMETER_COLUMNS = [column for column in requested_parameter_columns if column in scan_schema]
selected_cols = list(dict.fromkeys(IDENTIFIER_COLUMNS + PARAMETER_COLUMNS))

if unavailable_parameter_columns:
    print(f"Unavailable optional parameters: {len(unavailable_parameter_columns)}")

select_para_df = cda.select(*[F.col(f"`{column}`") for column in selected_cols]).orderBy(
    F.col("Timestamp"), F.col("AssetIdentifier").cast("string")
)
MAX_PANDAS_ROWS = 1_000_000
selected_row_count = select_para_df.count()
if selected_row_count > MAX_PANDAS_ROWS:
    raise RuntimeError(
        f"The selected range contains {selected_row_count:,} samples across all engines. "
        f"Reduce the date range below {MAX_PANDAS_ROWS:,} samples before converting to Pandas."
    )
_data_df = select_para_df.toPandas()
if not _data_df.empty:
    _data_df["Timestamp"] = pd.to_datetime(_data_df["Timestamp"], errors="coerce")
    _data_df["StartDatetime"] = pd.to_datetime(_data_df["StartDatetime"], errors="coerce")
    _data_df["AssetIdentifier"] = _data_df["AssetIdentifier"].astype("string")
    _data_df = _data_df.sort_values(
        ["Timestamp", "AssetIdentifier", "StartDatetime"], kind="stable"
    ).reset_index(drop=True)

print(f"Loaded samples across all ESNs: {len(_data_df):,}")
print(f"Loaded parameter columns: {len(PARAMETER_COLUMNS)}")
print(f"Loaded ESNs: {', '.join(selected_esns) if selected_esns else 'none'}")
print(
    "Flight-phase columns retained: "
    + (", ".join(f"{key}={value}" for key, value in FLIGHT_PHASE_COLUMNS.items()) or "none")
)

In [ ]:
def _short_parameter_name(column):
    return column[len(PARAMETER_PREFIX):] if column.startswith(PARAMETER_PREFIX) else column


combined_channel_columns = [_short_parameter_name(column) for column in PARAMETER_COLUMNS]
default_feature_columns = [
    _short_parameter_name(column) for column in CONTEXT_COLUMNS if column in PARAMETER_COLUMNS
]
previous_selected_parameters = globals().get("selected_parameters", [])
FEATURE_COLUMNS = [
    parameter for parameter in previous_selected_parameters if parameter in combined_channel_columns
]
if not FEATURE_COLUMNS:
    FEATURE_COLUMNS = default_feature_columns

input_search = widgets.Combobox(
    placeholder="Search and add a parameter",
    options=[option for option in combined_channel_columns if option not in FEATURE_COLUMNS],
    description="Add input:",
    ensure_option=True,
    layout=widgets.Layout(width="650px"),
)
input_display = widgets.SelectMultiple(
    options=tuple(FEATURE_COLUMNS),
    description="Plot inputs:",
    layout=widgets.Layout(width="650px", height="280px"),
)
remove_input_btn = widgets.Button(
    description="Remove highlighted",
    button_style="danger",
    icon="trash",
)
apply_input_btn = widgets.Button(
    description="Verify inputs",
    button_style="info",
)
selected_output = widgets.Output()
selected_parameters = list(FEATURE_COLUMNS)


def _refresh_search_options():
    active = set(input_display.options)
    input_search.options = [
        option for option in combined_channel_columns if option not in active
    ]


def _sync_selected_parameters():
    selected_parameters[:] = list(input_display.options)


def add_to_input(change):
    value = change["new"]
    if value and value in combined_channel_columns and value not in input_display.options:
        input_display.options = tuple(input_display.options) + (value,)
        _sync_selected_parameters()
        input_search.value = ""
        _refresh_search_options()


def remove_input(_):
    highlighted = set(input_display.value)
    input_display.options = tuple(
        option for option in input_display.options if option not in highlighted
    )
    _sync_selected_parameters()
    _refresh_search_options()


def apply_inputs(_):
    _sync_selected_parameters()
    with selected_output:
        selected_output.clear_output()
        print(f"Verified {len(selected_parameters)} plot inputs.")


input_search.observe(add_to_input, names="value")
remove_input_btn.on_click(remove_input)
apply_input_btn.on_click(apply_inputs)
input_column = widgets.VBox(
    [input_search, input_display, remove_input_btn, apply_input_btn, selected_output]
)
ipy_display(input_column)

In [ ]:
selected_parameters = globals().get("selected_parameters", [])
data_df = _data_df.copy()
if "Timestamp" in data_df.columns:
    data_df["Timestamp"] = pd.to_datetime(data_df["Timestamp"], errors="coerce")
if "StartDatetime" in data_df.columns:
    data_df["StartDatetime"] = pd.to_datetime(data_df["StartDatetime"], errors="coerce")
data_df = data_df.dropna(subset=["Timestamp", "StartDatetime", "AssetIdentifier"]).sort_values(
    ["Timestamp", "AssetIdentifier", "StartDatetime"], kind="stable"
)

MAX_SLIDER_POINTS = 1000
MAX_SIGNAL_GROUPS = 16
NO_DATA_COLOR = "#c62828"
CHANNEL_LINESTYLES = {"AC": "-", "BC": "--"}
engine_color_map = {
    esn: plt.get_cmap("tab10")(index % 10)
    for index, esn in enumerate(selected_esns)
}


def _full_parameter_name(short_name):
    return short_name if short_name.startswith(PARAMETER_PREFIX) else PARAMETER_PREFIX + short_name


def _channel_and_signal(short_name):
    for channel in CHANNELS:
        prefix = f"{channel}_"
        if short_name.startswith(prefix):
            return channel, short_name[len(prefix):]
    return "", short_name


def _classify_plot_values(series_list):
    combined = pd.concat(series_list, ignore_index=True) if series_list else pd.Series(dtype="object")
    non_null = combined.dropna()
    if non_null.empty:
        return {"kind": "empty", "discrete": False, "mapping": None}
    if non_null.map(lambda value: isinstance(value, (bool, np.bool_))).all():
        return {"kind": "boolean", "discrete": True, "mapping": None}
    if pd.api.types.is_datetime64_any_dtype(combined):
        return {"kind": "datetime", "discrete": False, "mapping": None}
    numeric = pd.to_numeric(non_null, errors="coerce")
    if numeric.notna().sum() >= max(1, math.ceil(0.8 * len(non_null))):
        values = numeric.dropna().to_numpy(dtype=float)
        discrete = bool(
            len(values) and len(np.unique(values)) <= 8 and np.allclose(values, np.round(values))
        )
        return {"kind": "numeric", "discrete": discrete, "mapping": None}
    native_datetime = non_null.map(
        lambda value: isinstance(value, (date, datetime, pd.Timestamp, np.datetime64))
    ).all()
    text_datetime = non_null.astype(str).str.contains(
        r"(?:\d{4}[-/]\d{1,2}[-/]\d{1,2})|(?:\d{1,2}:\d{2}:\d{2})",
        regex=True,
    ).mean() >= 0.8
    if native_datetime or text_datetime:
        parsed_datetime = pd.to_datetime(non_null, errors="coerce")
        if parsed_datetime.notna().sum() >= max(1, math.ceil(0.8 * len(non_null))):
            return {"kind": "datetime", "discrete": False, "mapping": None}
    labels = sorted(non_null.astype(str).unique())
    if len(labels) <= 20:
        return {
            "kind": "categorical",
            "discrete": True,
            "mapping": {label: index for index, label in enumerate(labels)},
        }
    return {"kind": "unsupported", "discrete": False, "mapping": None}


def _convert_plot_values(series, classification):
    kind = classification["kind"]
    if kind == "boolean":
        return series.map({True: 1.0, False: 0.0})
    if kind == "numeric":
        return pd.to_numeric(series, errors="coerce")
    if kind == "datetime":
        parsed = pd.to_datetime(series, errors="coerce")
        converted = pd.Series(np.nan, index=series.index, dtype="float64")
        valid = parsed.notna()
        if valid.any():
            converted.loc[valid] = mdates.date2num(parsed.loc[valid].dt.to_pydatetime())
        return converted
    if kind == "categorical":
        mapping = classification["mapping"]
        return series.map(lambda value: np.nan if pd.isna(value) else mapping.get(str(value), np.nan))
    return pd.Series(np.nan, index=series.index, dtype="float64")


def _match_timestamp_timezone(value, reference):
    timestamp = pd.Timestamp(value)
    if reference.tzinfo is None and timestamp.tzinfo is not None:
        return timestamp.tz_localize(None)
    if reference.tzinfo is not None and timestamp.tzinfo is None:
        return timestamp.tz_localize(reference.tzinfo)
    if reference.tzinfo is not None and timestamp.tzinfo is not None:
        return timestamp.tz_convert(reference.tzinfo)
    return timestamp


def _format_gap_duration(total_seconds):
    total_seconds = max(0, int(round(total_seconds)))
    days, remainder = divmod(total_seconds, 86400)
    hours, remainder = divmod(remainder, 3600)
    minutes, seconds = divmod(remainder, 60)
    if days:
        return f"{days}d {hours:02d}h"
    if hours:
        return f"{hours}h {minutes:02d}m"
    if minutes:
        return f"{minutes}m {seconds:02d}s"
    return f"{seconds}s"


def _build_compressed_timeline(frame):
    flight_frames = []
    for flight_key, flight_frame in frame.groupby("StartDatetime", sort=False, dropna=False):
        ordered = flight_frame.sort_values(["Timestamp", "AssetIdentifier"], kind="stable").copy()
        actual_start = ordered["Timestamp"].min()
        actual_end = ordered["Timestamp"].max()
        positive_deltas = ordered["Timestamp"].drop_duplicates().sort_values().diff().dt.total_seconds()
        positive_deltas = positive_deltas.loc[positive_deltas > 0]
        cadence_seconds = float(positive_deltas.median()) if not positive_deltas.empty else 1.0
        flight_frames.append(
            {
                "key": flight_key,
                "frame": ordered,
                "actual_start": actual_start,
                "actual_end": actual_end,
                "cadence_seconds": max(cadence_seconds, 0.001),
                "duration_seconds": max(
                    float((actual_end - actual_start).total_seconds()), max(cadence_seconds, 1.0)
                ),
            }
        )
    flight_frames.sort(key=lambda item: item["actual_start"])
    active_seconds = sum(item["duration_seconds"] for item in flight_frames)
    gap_count = max(0, len(flight_frames) - 1)
    visual_gap_seconds = (
        max(active_seconds * min(0.02, 0.15 / gap_count), 1.0) if gap_count else 0.0
    )
    cursor = 0.0
    timeline_parts = []
    flights = []
    gaps = []
    previous = None
    for item in flight_frames:
        if previous is not None:
            gap_start = cursor
            cursor += visual_gap_seconds
            gaps.append(
                {
                    "plot_start": gap_start,
                    "plot_end": cursor,
                    "plot_center": (gap_start + cursor) / 2.0,
                    "actual_seconds": max(
                        0.0, float((item["actual_start"] - previous["actual_end"]).total_seconds())
                    ),
                }
            )
        plot_start = cursor
        part = item["frame"].copy()
        part["_plot_x"] = plot_start + (part["Timestamp"] - item["actual_start"]).dt.total_seconds()
        plot_end = plot_start + item["duration_seconds"]
        part["_flight_order"] = len(flights)
        timeline_parts.append(part)
        flights.append({**item, "plot_start": plot_start, "plot_end": plot_end})
        cursor = plot_end
        previous = item
    timeline = pd.concat(timeline_parts, ignore_index=True) if timeline_parts else frame.iloc[0:0].copy()
    return timeline, flights, gaps, cursor


def _timeline_ticks(flights, maximum_ticks=14):
    if not flights:
        return [], []
    if len(flights) > 7:
        indices = np.unique(
            np.linspace(0, len(flights) - 1, num=min(maximum_ticks, len(flights)), dtype=int)
        )
        specs = [(flights[index], 0.0) for index in indices]
    else:
        ticks_per_flight = max(1, maximum_ticks // len(flights))
        specs = [
            (flight, float(fraction))
            for flight in flights
            for fraction in np.linspace(0.0, 1.0, num=ticks_per_flight, endpoint=True)
        ]
    positions = []
    labels = []
    for flight, fraction in specs:
        actual_seconds = max(
            0.0, float((flight["actual_end"] - flight["actual_start"]).total_seconds())
        )
        actual_time = flight["actual_start"] + pd.to_timedelta(actual_seconds * fraction, unit="s")
        positions.append(flight["plot_start"] + flight["duration_seconds"] * fraction)
        labels.append(actual_time.strftime("%d %b %Y\n%H:%M:%S"))
    return positions, labels


def _draw_axis_break(axis, gap, show_label=False):
    transform = blended_transform_factory(axis.transData, axis.transAxes)
    half_width = max((gap["plot_end"] - gap["plot_start"]) * 0.32, 0.25)
    center = gap["plot_center"]
    axis.plot(
        np.linspace(center - half_width, center + half_width, 7),
        np.array([-0.018, 0.018, -0.018, 0.018, -0.018, 0.018, -0.018]),
        color="#333333",
        linewidth=1.3,
        transform=transform,
        clip_on=False,
        zorder=8,
    )
    axis.axvline(center, color="#777777", linewidth=0.7, linestyle=":", alpha=0.45, zorder=1)
    if show_label:
        axis.annotate(
            f"Gap\n{_format_gap_duration(gap['actual_seconds'])}",
            xy=(center, 0),
            xycoords=transform,
            xytext=(0, -42),
            textcoords="offset points",
            ha="center",
            va="top",
            fontsize=8,
            color="#444444",
            annotation_clip=False,
        )


def _plot_parameters(start_time, end_time):
    chosen = list(selected_parameters)
    if not chosen:
        print("No plot inputs are applied.")
        return
    grouped = OrderedDict()
    for short_name in chosen:
        channel, signal = _channel_and_signal(short_name)
        grouped.setdefault(signal, []).append((channel, short_name))
    if len(grouped) > MAX_SIGNAL_GROUPS:
        print(f"Reduce the selection to {MAX_SIGNAL_GROUPS} signal groups or fewer.")
        return
    visible = data_df.loc[data_df["Timestamp"].between(start_time, end_time, inclusive="both")].copy()
    if visible.empty:
        print("No samples fall inside the requested time range.")
        return
    timeline, flights, gaps, timeline_end = _build_compressed_timeline(visible)
    if timeline.empty:
        print("No valid flight timestamps fall inside the requested time range.")
        return
    esns_in_view = [
        esn for esn in selected_esns if esn in set(timeline["AssetIdentifier"].astype(str))
    ]
    figure, axes = plt.subplots(
        len(grouped),
        1,
        figsize=(22, max(5.5, 3.2 * len(grouped))),
        sharex=True,
        squeeze=False,
    )
    skipped = []
    for axis, (signal, members) in zip(axes[:, 0], grouped.items()):
        available_members = [
            (channel, short_name, _full_parameter_name(short_name))
            for channel, short_name in members
            if _full_parameter_name(short_name) in timeline.columns
        ]
        classification = _classify_plot_values(
            [timeline[full_name] for _, _, full_name in available_members]
        )
        plotted = False
        for channel, short_name, full_name in available_members:
            for esn in esns_in_view:
                parameter_frame = timeline.loc[
                    timeline["AssetIdentifier"].astype(str) == str(esn),
                    ["Timestamp", "StartDatetime", "_plot_x", "_flight_order", full_name],
                ].copy()
                parameter_frame["_value"] = _convert_plot_values(
                    parameter_frame[full_name], classification
                )
                parameter_frame = parameter_frame.dropna(subset=["_value"])
                if parameter_frame.empty:
                    skipped.append(f"ESN {esn} {short_name}")
                    continue
                label_pending = True
                previous_flight_endpoint = None
                for flight_order, segment in parameter_frame.groupby("_flight_order", sort=True):
                    segment = segment.sort_values("Timestamp", kind="stable")
                    deltas = segment["Timestamp"].diff().dt.total_seconds()
                    median_cadence = flights[int(flight_order)]["cadence_seconds"]
                    chunk_ids = deltas.gt(max(3.0 * median_cadence, 5.0)).cumsum()
                    chunks = [chunk for _, chunk in segment.groupby(chunk_ids, sort=True)]
                    if previous_flight_endpoint is not None:
                        first_row = chunks[0].iloc[0]
                        axis.plot(
                            [previous_flight_endpoint[0], first_row["_plot_x"]],
                            [previous_flight_endpoint[1], first_row["_value"]],
                            color=NO_DATA_COLOR,
                            linewidth=1.2,
                            linestyle=(0, (2, 3)),
                            zorder=3,
                        )
                    for chunk_index, chunk in enumerate(chunks):
                        if chunk_index:
                            previous_row = chunks[chunk_index - 1].iloc[-1]
                            next_row = chunk.iloc[0]
                            axis.plot(
                                [previous_row["_plot_x"], next_row["_plot_x"]],
                                [previous_row["_value"], next_row["_value"]],
                                color=NO_DATA_COLOR,
                                linewidth=1.2,
                                linestyle=(0, (2, 3)),
                                zorder=3,
                            )
                        axis.plot(
                            chunk["_plot_x"],
                            chunk["_value"],
                            color=engine_color_map.get(esn, "#333333"),
                            linestyle=CHANNEL_LINESTYLES.get(channel, "-"),
                            linewidth=1.25,
                            drawstyle="steps-post" if classification["discrete"] else "default",
                            label=f"ESN {esn} {channel or short_name}" if label_pending else "_nolegend_",
                            zorder=4,
                        )
                        label_pending = False
                    last_row = chunks[-1].iloc[-1]
                    previous_flight_endpoint = (last_row["_plot_x"], last_row["_value"])
                    plotted = True
        axis.set_ylabel(signal, fontsize=9)
        axis.grid(True, alpha=0.22, zorder=1)
        axis.margins(y=0.08)
        if classification["kind"] == "categorical":
            inverse = {value: key for key, value in classification["mapping"].items()}
            axis.set_yticks(sorted(inverse))
            axis.set_yticklabels([inverse[value] for value in sorted(inverse)], fontsize=8)
        elif classification["kind"] == "datetime":
            axis.yaxis_date()
            axis.yaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d\n%H:%M:%S"))
        if axis is not axes[-1, 0]:
            for gap in gaps:
                _draw_axis_break(axis, gap)
        if plotted:
            axis.legend(loc="upper right", ncol=min(4, max(1, len(esns_in_view) * len(members))), framealpha=0.85)
        else:
            axis.text(0.5, 0.5, "No plottable samples", ha="center", va="center", transform=axis.transAxes)
    bottom_axis = axes[-1, 0]
    tick_positions, tick_labels = _timeline_ticks(flights)
    bottom_axis.set_xticks(tick_positions)
    bottom_axis.set_xticklabels(tick_labels, rotation=25, ha="right", fontsize=8)
    bottom_axis.set_xlabel(
        "Flight time - calendar timestamps shown; inter-flight gaps compressed",
        labelpad=48 if gaps else 12,
    )
    for gap in gaps:
        _draw_axis_break(bottom_axis, gap, show_label=True)
    for axis in axes[:, 0]:
        axis.set_xlim(0.0, max(timeline_end, 1.0))
    figure.legend(
        handles=[
            Line2D([0], [0], color=NO_DATA_COLOR, linewidth=1.5, linestyle=(0, (2, 3)), label="No recorded data")
        ],
        loc="lower center",
        bbox_to_anchor=(0.5, 0.012),
        frameon=False,
    )
    figure.suptitle(
        f"Aircraft {selected_acid} | {len(esns_in_view)} ESNs | {len(flights)} flight windows | "
        f"{start_time} through {end_time}",
        fontsize=14,
    )
    figure.tight_layout(rect=(0.02, 0.105, 0.99, 0.95))
    plt.show()
    plt.close(figure)
    if skipped:
        print("Skipped series without plottable samples: " + ", ".join(sorted(set(skipped))))


if data_df.empty:
    print("No samples are available for the selected flights.")
else:
    all_timestamps = pd.DatetimeIndex(data_df["Timestamp"].drop_duplicates().sort_values())
    if len(all_timestamps) > MAX_SLIDER_POINTS:
        indices = np.linspace(0, len(all_timestamps) - 1, num=MAX_SLIDER_POINTS, dtype=int)
        slider_timestamps = [all_timestamps[index] for index in np.unique(indices)]
    else:
        slider_timestamps = list(all_timestamps)

    def _timestamp_label(timestamp):
        return pd.Timestamp(timestamp).strftime("%Y-%m-%d %H:%M:%S.%f").rstrip("0").rstrip(".")

    def _nearest_recorded_timestamp(timestamp):
        matched = _match_timestamp_timezone(timestamp, all_timestamps[0])
        index = all_timestamps.get_indexer([matched], method="nearest")[0]
        return pd.Timestamp(all_timestamps[index])

    time_slider = SelectionRangeSlider(
        options=[(_timestamp_label(timestamp), timestamp) for timestamp in slider_timestamps],
        index=(0, len(slider_timestamps) - 1),
        description="Time range:",
        orientation="horizontal",
        layout=widgets.Layout(width="1200px"),
        continuous_update=False,
    )
    txt_start = widgets.Text(
        value=_timestamp_label(all_timestamps[0]),
        description="Start:",
        layout=widgets.Layout(width="500px"),
    )
    txt_end = widgets.Text(
        value=_timestamp_label(all_timestamps[-1]),
        description="End:",
        layout=widgets.Layout(width="500px"),
    )
    plot_btn = widgets.Button(
        description="Plot",
        button_style="success",
        icon="line-chart",
        layout=widgets.Layout(width="150px"),
    )
    plot_output = widgets.Output()

    def _render_range(start_time, end_time):
        with plot_output:
            clear_output(wait=True)
            if start_time > end_time:
                start_time, end_time = end_time, start_time
            _plot_parameters(start_time, end_time)

    def _on_plot_click(_):
        try:
            start_time = _nearest_recorded_timestamp(txt_start.value)
            end_time = _nearest_recorded_timestamp(txt_end.value)
        except Exception:
            with plot_output:
                clear_output(wait=True)
                print("Start and end must be valid date-time values.")
            return
        if start_time > end_time:
            start_time, end_time = end_time, start_time
        txt_start.value = _timestamp_label(start_time)
        txt_end.value = _timestamp_label(end_time)
        _render_range(start_time, end_time)

    def _on_slider_change(change):
        start_time, end_time = map(pd.Timestamp, change["new"])
        txt_start.value = _timestamp_label(start_time)
        txt_end.value = _timestamp_label(end_time)
        _render_range(start_time, end_time)

    plot_btn.on_click(_on_plot_click)
    time_slider.observe(_on_slider_change, names="value")
    ipy_display(widgets.VBox([widgets.VBox([txt_start, txt_end, time_slider, plot_btn]), plot_output]))
    _render_range(all_timestamps[0], all_timestamps[-1])

In [ ]:
TR_IS_DEPLOYED_COLUMNS = {
    channel: f"{PARAMETER_PREFIX}{channel}_IThrustReverser_trIsDeployed_data"
    for channel in CHANNELS
}

existing_comparison_date_widget = globals().get("comparison_date_widget")
previous_comparison_date = getattr(existing_comparison_date_widget, "value", None)
comparison_date_widget = widgets.DatePicker(
    value=previous_comparison_date or end_date_widget.value,
    description="Compare day:",
    layout=widgets.Layout(width="300px"),
)
run_transition_btn = widgets.Button(
    description="Build transition table",
    button_style="primary",
    layout=widgets.Layout(width="230px"),
)
transition_output = widgets.Output()


def _normalized_state_expression(column_name):
    raw_column = F.col(f"`{column_name}`")
    raw_text = F.lower(F.trim(raw_column.cast("string")))
    numeric_value = raw_column.cast("double")
    return (
        F.when(raw_text.isin("true", "on", "yes"), F.lit(1))
        .when(raw_text.isin("false", "off", "no"), F.lit(0))
        .when(numeric_value == F.lit(1.0), F.lit(1))
        .when(numeric_value == F.lit(0.0), F.lit(0))
        .otherwise(F.lit(None).cast("integer"))
    )


def _millisecond_timestamp(value):
    timestamp = pd.Timestamp(value)
    return timestamp.strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]


def _render_coverage_table(coverage_records):
    header = "".join(
        f"<th>{html.escape(column)}</th>"
        for column in (
            "ESN",
            "Position",
            "Channel",
            "Rows",
            "Non-null",
            "Invalid",
            "Initializations",
            "Transitions",
            "Conflicts",
            "First sample",
            "Last sample",
        )
    )
    rows = []
    for record in coverage_records:
        values = [
            record["ESN"],
            record["Position"],
            record["Channel"],
            f"{record['Rows']:,}",
            f"{record['Non-null']:,}",
            f"{record['Invalid']:,}",
            f"{record['Initializations']:,}",
            f"{record['Transitions']:,}",
            f"{record['Conflicts']:,}",
            record["First sample"],
            record["Last sample"],
        ]
        rows.append("<tr>" + "".join(f"<td>{html.escape(str(value))}</td>" for value in values) + "</tr>")
    return (
        "<div style='overflow:auto;max-height:320px;border:1px solid #ccc'>"
        "<table class='coverage-table' style='width:100%;border-collapse:collapse;font-size:12px'>"
        "<thead><tr>" + header + "</tr></thead><tbody>" + "".join(rows) + "</tbody></table></div>"
        "<style>.coverage-table th,.coverage-table td{padding:6px 8px;border-bottom:1px solid #ddd;text-align:left}"
        ".coverage-table th{position:sticky;top:0;background:#f5f5f5;z-index:2}</style>"
    )


def _render_transition_table(event_rows, esns):
    columns = [(esn, channel) for esn in esns for channel in CHANNELS]
    rows_by_timestamp = OrderedDict()
    for event in event_rows:
        timestamp = pd.Timestamp(event["Timestamp"])
        row = rows_by_timestamp.setdefault(timestamp, {column: [] for column in columns})
        key = (str(event["ESN"]), str(event["Channel"]))
        if key not in row:
            continue
        if event["EventType"] == "conflict":
            raw_values = ", ".join(str(value) for value in event.get("RawValues", []))
            label = f"Conflict: {raw_values}"
        else:
            label = f"IsDeployed={int(event['State'])}"
        row[key].append((label, event["EventType"], event.get("State")))

    header_cells = ["<th>Timestamp</th>"] + [
        f"<th>ESN {html.escape(str(esn))} {channel}</th>" for esn, channel in columns
    ]
    body_rows = []
    for timestamp, values in rows_by_timestamp.items():
        cells = [
            f"<td class='transition-time' title='{html.escape(timestamp.isoformat())}'>"
            f"{html.escape(_millisecond_timestamp(timestamp))}</td>"
        ]
        for column in columns:
            entries = values[column]
            if not entries:
                cells.append("<td></td>")
                continue
            fragments = []
            classes = []
            for label, event_type, state in entries:
                fragments.append(html.escape(label))
                if event_type == "conflict":
                    classes.append("state-conflict")
                elif state == 1:
                    classes.append("state-one")
                else:
                    classes.append("state-zero")
                if event_type == "initialization":
                    classes.append("state-initial")
            cells.append(f"<td class='{' '.join(sorted(set(classes)))}'>{'<br>'.join(fragments)}</td>")
        body_rows.append("<tr>" + "".join(cells) + "</tr>")

    return """
    <style>
      .transition-scroll {width:100%;max-height:720px;overflow:auto;border:1px solid #999;background:white}
      .transition-ledger {width:100%;min-width:1000px;border-collapse:collapse;font-size:13px}
      .transition-ledger th {position:sticky;top:0;z-index:3;background:#f3f3f3;border:1px solid #aaa;padding:7px;text-align:left}
      .transition-ledger td {border:1px solid #bbb;padding:6px 8px;white-space:nowrap}
      .transition-time {font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;background:#fafafa}
      .state-one {background:#c6efce;color:#145a24;font-weight:700}
      .state-zero {background:#e7e6e6;color:#333;font-weight:700}
      .state-initial {box-shadow:inset 4px 0 0 #f9a825}
      .state-conflict {background:#ffc7ce;color:#9c0006;font-weight:800}
    </style>
    <div class="transition-scroll"><table class="transition-ledger"><thead><tr>
    """ + "".join(header_cells) + "</tr></thead><tbody>" + "".join(body_rows) + "</tbody></table></div>"


def _build_transition_comparison(_, raise_errors=False):
    run_transition_btn.disabled = True
    cached_scan = None
    try:
        with transition_output:
            clear_output(wait=True)
            chosen_aircraft = aircraft_widget.value
            chosen_day = comparison_date_widget.value
            if chosen_day is None:
                raise ValueError("A comparison day is required.")
            day_start, day_end = _calendar_bounds(chosen_day, chosen_day)
            master_df, windows_df, esns, positions, aircraft_ids = _resolve_engine_windows(
                chosen_aircraft, day_start, day_end
            )
            if not esns:
                print(f"WARNING: No engine master records overlap {chosen_day} for aircraft {chosen_aircraft}.")
                globals().update(
                    {
                        "transition_event_df": pd.DataFrame(),
                        "transition_coverage_df": pd.DataFrame(),
                        "transition_esns": [],
                    }
                )
                return

            scan_schema_columns = spark.table(SCAN_TABLE).columns
            available_channels = {
                channel: column
                for channel, column in TR_IS_DEPLOYED_COLUMNS.items()
                if column in scan_schema_columns
            }
            missing_channels = [channel for channel in CHANNELS if channel not in available_channels]
            for channel in missing_channels:
                print(f"WARNING: The {channel} trIsDeployed parameter is absent from the scan schema.")

            comparison_scan, _ = _windowed_scan(
                windows_df,
                aircraft_ids,
                esns,
                requested_columns=list(available_channels.values()),
                strict_start=day_start,
                strict_end=day_end,
            )
            cached_scan = comparison_scan.cache()
            total_rows = cached_scan.count()
            print(f"Aircraft {chosen_aircraft} | Day {chosen_day} | ESNs: {', '.join(esns)}")
            print(f"Strict calendar-day scan rows: {total_rows:,}")

            event_frames = []
            coverage_lookup = {}
            for channel, column_name in available_channels.items():
                raw_column = F.col(f"`{column_name}`")
                normalized_state = _normalized_state_expression(column_name)
                channel_rows = cached_scan.select(
                    F.col("Timestamp"),
                    F.col("StartDatetime"),
                    F.col("AssetIdentifier").cast("string").alias("ESN"),
                    raw_column.cast("string").alias("RawValue"),
                    normalized_state.alias("State"),
                )
                coverage_rows = (
                    channel_rows.groupBy("ESN")
                    .agg(
                        F.count(F.lit(1)).alias("Rows"),
                        F.count("RawValue").alias("NonNull"),
                        F.sum(
                            F.when(F.col("RawValue").isNotNull() & F.col("State").isNull(), 1).otherwise(0)
                        ).alias("Invalid"),
                        F.min(F.when(F.col("RawValue").isNotNull(), F.col("Timestamp"))).alias("FirstSample"),
                        F.max(F.when(F.col("RawValue").isNotNull(), F.col("Timestamp"))).alias("LastSample"),
                    )
                    .collect()
                )
                for row in coverage_rows:
                    coverage_lookup[(str(row.ESN), channel)] = {
                        "Rows": int(row.Rows or 0),
                        "Non-null": int(row.NonNull or 0),
                        "Invalid": int(row.Invalid or 0),
                        "First sample": _millisecond_timestamp(row.FirstSample) if row.FirstSample else "",
                        "Last sample": _millisecond_timestamp(row.LastSample) if row.LastSample else "",
                    }

                grouped = channel_rows.where(F.col("RawValue").isNotNull()).groupBy(
                    "Timestamp", "StartDatetime", "ESN"
                ).agg(
                    F.collect_set("State").alias("States"),
                    F.collect_set("RawValue").alias("RawValues"),
                )
                valid_unique = grouped.where(F.size("States") == 1).select(
                    "Timestamp",
                    "StartDatetime",
                    "ESN",
                    F.element_at("States", 1).cast("integer").alias("State"),
                    "RawValues",
                )
                sequence_window = Window.partitionBy("ESN", "StartDatetime").orderBy("Timestamp")
                sequenced = valid_unique.withColumn("PreviousState", F.lag("State").over(sequence_window))
                events = sequenced.where(
                    F.col("PreviousState").isNull() | (F.col("State") != F.col("PreviousState"))
                ).select(
                    "Timestamp",
                    "StartDatetime",
                    "ESN",
                    F.lit(channel).alias("Channel"),
                    "State",
                    F.when(F.col("PreviousState").isNull(), F.lit("initialization"))
                    .otherwise(F.lit("transition"))
                    .alias("EventType"),
                    "RawValues",
                )
                conflicts = grouped.where(F.size("States") > 1).select(
                    "Timestamp",
                    "StartDatetime",
                    "ESN",
                    F.lit(channel).alias("Channel"),
                    F.lit(None).cast("integer").alias("State"),
                    F.lit("conflict").alias("EventType"),
                    "RawValues",
                )
                event_frames.extend([events, conflicts])

            if event_frames:
                all_events = event_frames[0]
                for frame in event_frames[1:]:
                    all_events = all_events.unionByName(frame)
                event_pdf = all_events.orderBy("Timestamp", "ESN", "Channel").toPandas()
            else:
                event_pdf = pd.DataFrame(
                    columns=["Timestamp", "StartDatetime", "ESN", "Channel", "State", "EventType", "RawValues"]
                )

            event_counts = {}
            if not event_pdf.empty:
                for (esn, channel, event_type), count in event_pdf.groupby(
                    ["ESN", "Channel", "EventType"], dropna=False
                ).size().items():
                    event_counts[(str(esn), str(channel), str(event_type))] = int(count)

            coverage_records = []
            warnings = []
            for esn in esns:
                for channel in CHANNELS:
                    base = coverage_lookup.get(
                        (str(esn), channel),
                        {"Rows": 0, "Non-null": 0, "Invalid": 0, "First sample": "", "Last sample": ""},
                    )
                    record = {
                        "ESN": str(esn),
                        "Position": positions.get(esn) or "",
                        "Channel": channel,
                        **base,
                        "Initializations": event_counts.get((str(esn), channel, "initialization"), 0),
                        "Transitions": event_counts.get((str(esn), channel, "transition"), 0),
                        "Conflicts": event_counts.get((str(esn), channel, "conflict"), 0),
                    }
                    coverage_records.append(record)
                    if channel not in available_channels:
                        warnings.append(f"ESN {esn} {channel}: parameter missing from schema")
                    elif record["Non-null"] == 0:
                        warnings.append(f"ESN {esn} {channel}: no non-null samples")
                    if record["Invalid"]:
                        warnings.append(
                            f"ESN {esn} {channel}: {record['Invalid']:,} values could not be normalized to 0 or 1"
                        )
                    if record["Conflicts"]:
                        warnings.append(
                            f"ESN {esn} {channel}: {record['Conflicts']:,} timestamps contain conflicting values"
                        )

            for warning in warnings:
                print("WARNING: " + warning)
            ipy_display(HTML(_render_coverage_table(coverage_records)))
            if event_pdf.empty:
                print("WARNING: No initialization or transition events were found for the selected day.")
            else:
                event_records = event_pdf.to_dict("records")
                ipy_display(HTML(_render_transition_table(event_records, esns)))
                print(f"Transition-ledger timestamps: {event_pdf['Timestamp'].nunique():,}")

            globals().update(
                {
                    "transition_event_df": event_pdf,
                    "transition_coverage_df": pd.DataFrame(coverage_records),
                    "transition_esns": esns,
                    "transition_master_df": master_df,
                    "transition_windows_df": windows_df,
                }
            )
    except Exception as exc:
        with transition_output:
            print(f"Unable to build the transition table: {exc}")
        if raise_errors:
            raise
    finally:
        if cached_scan is not None:
            cached_scan.unpersist()
        run_transition_btn.disabled = False


run_transition_btn.on_click(_build_transition_comparison)
ipy_display(widgets.VBox([widgets.HBox([comparison_date_widget, run_transition_btn]), transition_output]))
_build_transition_comparison(None, raise_errors=True)

In [ ]:
schema_columns = set(cda_cols)
selected_parameter_set = set(selected_parameters)
channel_pair_records = []
for suffix in THRUST_REVERSER_SUFFIXES:
    ac_short = f"AC_{suffix}"
    bc_short = f"BC_{suffix}"
    ac_full = PARAMETER_PREFIX + ac_short
    bc_full = PARAMETER_PREFIX + bc_short
    channel_pair_records.append(
        {
            "Signal": suffix,
            "AC available": ac_full in schema_columns,
            "BC available": bc_full in schema_columns,
            "AC selected": ac_short in selected_parameter_set,
            "BC selected": bc_short in selected_parameter_set,
        }
    )

channel_pair_df = pd.DataFrame(channel_pair_records)
unpaired_schema_rows = channel_pair_df.loc[
    channel_pair_df["AC available"] != channel_pair_df["BC available"]
]
print(f"Thrust-reverser signal pairs: {len(channel_pair_df)}")
print(f"Schema pairs with unequal AC/BC availability: {len(unpaired_schema_rows)}")
ipy_display(channel_pair_df)

In [ ]:
diagnostic_master_columns = [
    "AircraftIdentifier",
    "EngineSerialNumber",
    "AircraftId",
    "EngineId",
    "StartDatetime",
    "EndDatetime",
    "Duration",
    "EnginePosition",
    "OperatorId",
    "OperatorCode",
    "LastGeneratedDatetime",
    "FirstGeneratedDatetime",
    "Changed",
    "Created",
    "Migrated",
    "CalendarId",
]
diagnostic_master_with_duration = cda_master_dt.withColumn(
    "Duration", F.col("EndDatetime") - F.col("StartDatetime")
)
diagnostic_available_columns = [
    column for column in diagnostic_master_columns if column in diagnostic_master_with_duration.columns
]
diagnostic_master_df = diagnostic_master_with_duration.select(*diagnostic_available_columns).orderBy(
    F.col("StartDatetime").desc(), F.col("EngineSerialNumber"), F.col("EndDatetime").desc()
)
print(
    f"Flight inventory for aircraft {selected_acid}, all overlapping ESNs, "
    f"from {window_start_date} through {window_end_date}"
)
display(diagnostic_master_df)

In [ ]:
if flight_available:
    diagnostic_selected_master_df = flight_windows_df.withColumn(
        "Duration", F.col("EndDatetime") - F.col("StartDatetime")
    ).orderBy(F.col("StartDatetime"), F.col("EngineSerialNumber"))
    print(
        f"Selected engine-flight windows: {len(flight_rows)} from "
        f"{window_start_date} through {window_end_date}"
    )
else:
    diagnostic_selected_master_df = flight_windows_df.limit(0)
    print("No selected flight windows are available.")
display(diagnostic_selected_master_df)

In [ ]:
def _disagreement_summary(ac_values, bc_values):
    both_present = ac_values.notna() & bc_values.notna()
    ac_present = ac_values.loc[both_present]
    bc_present = bc_values.loc[both_present]
    if ac_present.empty:
        return both_present, pd.Series(dtype="bool"), "No overlap"
    if pd.api.types.is_datetime64_any_dtype(ac_present) or pd.api.types.is_datetime64_any_dtype(bc_present):
        disagreements = pd.to_datetime(ac_present, errors="coerce").ne(
            pd.to_datetime(bc_present, errors="coerce")
        )
        return both_present, disagreements.fillna(False), "Exact datetime"
    ac_numeric = pd.to_numeric(ac_present, errors="coerce")
    bc_numeric = pd.to_numeric(bc_present, errors="coerce")
    numeric_mask = ac_numeric.notna() & bc_numeric.notna()
    combined_numeric = pd.concat([ac_numeric.loc[numeric_mask], bc_numeric.loc[numeric_mask]])
    is_discrete = bool(
        not combined_numeric.empty
        and combined_numeric.nunique() <= 8
        and np.allclose(combined_numeric.to_numpy(dtype=float), np.round(combined_numeric.to_numpy(dtype=float)))
    )
    disagreements = pd.Series(False, index=ac_present.index, dtype="bool")
    if numeric_mask.any():
        if is_discrete:
            disagreements.loc[numeric_mask] = ac_numeric.loc[numeric_mask].ne(bc_numeric.loc[numeric_mask])
        else:
            scale = np.maximum(
                ac_numeric.loc[numeric_mask].abs().to_numpy(dtype=float),
                bc_numeric.loc[numeric_mask].abs().to_numpy(dtype=float),
            )
            difference = (
                ac_numeric.loc[numeric_mask] - bc_numeric.loc[numeric_mask]
            ).abs().to_numpy(dtype=float)
            disagreements.loc[numeric_mask] = difference > (0.05 * scale + 1e-12)
    non_numeric_mask = ~numeric_mask
    if non_numeric_mask.any():
        disagreements.loc[non_numeric_mask] = ac_present.loc[non_numeric_mask].astype(str).ne(
            bc_present.loc[non_numeric_mask].astype(str)
        )
    mode = "Exact discrete/object" if is_discrete else "5% symmetric continuous; exact object"
    return both_present, disagreements.fillna(False), mode


channel_coverage_records = []
for esn in selected_esns:
    engine_frame = _data_df.loc[_data_df["AssetIdentifier"].astype(str) == str(esn)]
    for suffix in THRUST_REVERSER_SUFFIXES:
        ac_column = f"{PARAMETER_PREFIX}AC_{suffix}"
        bc_column = f"{PARAMETER_PREFIX}BC_{suffix}"
        if ac_column not in engine_frame.columns or bc_column not in engine_frame.columns:
            continue
        ac_values = engine_frame[ac_column]
        bc_values = engine_frame[bc_column]
        both_present, disagreements, comparison_mode = _disagreement_summary(ac_values, bc_values)
        channel_coverage_records.append(
            {
                "ESN": esn,
                "Position": engine_positions.get(esn),
                "Signal": suffix,
                "AC non-null": int(ac_values.notna().sum()),
                "BC non-null": int(bc_values.notna().sum()),
                "Both non-null": int(both_present.sum()),
                "Disagreements": int(disagreements.sum()),
                "Comparison mode": comparison_mode,
            }
        )

channel_coverage_df = pd.DataFrame(channel_coverage_records)
if channel_coverage_df.empty:
    print("No paired AC/BC samples are available for coverage analysis.")
else:
    print(f"Channel coverage calculated from {len(_data_df):,} samples across {len(selected_esns)} ESNs.")
ipy_display(channel_coverage_df)

In [ ]:
preview_identifier_columns = [
    column for column in IDENTIFIER_COLUMNS if column in select_para_df.columns
]
preview_parameter_columns = [
    _full_parameter_name(short_name)
    for short_name in selected_parameters
    if _full_parameter_name(short_name) in select_para_df.columns
]
preview_columns = preview_identifier_columns + [
    column for column in preview_parameter_columns if column not in preview_identifier_columns
]
diagnostic_preview_df = select_para_df.select(*preview_columns).orderBy(
    F.col("Timestamp"), F.col("AssetIdentifier").cast("string")
).limit(50)
print(
    f"First 50 ordered samples for aircraft {selected_acid}, {len(selected_esns)} ESNs, "
    f"{len(flight_rows)} engine-flight windows from {window_start_date} through {window_end_date}"
)
display(diagnostic_preview_df)